# Atacama (ATAMOS, Fixed_PERC)

## Paths

In [ ]:
from pathlib import Path

ROOT = Path("../..").resolve()
RAW_DIR = ROOT / "data" / "raw"
OUT_DIR = ROOT / "data" / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import os
import pandas as pd

ATAMOS_DIR = None
for root, dirs, files in os.walk(str(RAW_DIR)):
    if os.path.basename(root) == 'ATAMOS':
        ATAMOS_DIR = root
        break

if ATAMOS_DIR is None:
    raise FileNotFoundError("ATAMOS folder not found under data/raw.")

In [ ]:
PERC_PATH = os.path.join(ATAMOS_DIR, 'Fixed_PERC', 'Fixed_PERC.csv')

## Build unified hourly dataset

In [ ]:
import numpy as np

df_raw = pd.read_csv(PERC_PATH, sep=';', on_bad_lines='skip', engine='python')

df_raw['Fecha'] = pd.to_datetime(df_raw['Fecha']).dt.tz_localize(None)
df_raw.set_index('Fecha', inplace=True)

df_raw['Power_W'] = df_raw['Vmpp(V)'] * df_raw['Impp(A)']

df_hourly = df_raw.resample('1h').mean().reset_index()

df_hourly = df_hourly.iloc[:8760].copy()

df_atacama = pd.DataFrame()
df_atacama['utctime'] = df_hourly['Fecha']
df_atacama['Power_W'] = df_hourly['Power_W']

P_MAX_INSTALLED = 1408.0
df_atacama['Power_PU'] = np.clip(df_hourly['Power_W'] / P_MAX_INSTALLED, 0.0, 1.0)

df_atacama['Irradiance_POA'] = df_hourly['Front GPOA (W/m²)'].fillna(df_hourly['GHI (W/m²)'])
df_atacama['GHI'] = df_hourly['GHI (W/m²)'].fillna(0.0)
df_atacama['Temp_Ambient'] = df_hourly['Amb. Temp. (°C)']
df_atacama['Wind_Speed'] = df_hourly['Wind Speed (m/s)']
df_atacama['Humidity'] = df_hourly['Humidity(%)']

hours = df_atacama['utctime'].dt.hour
months = df_atacama['utctime'].dt.month
df_atacama['HOUR_SIN'] = np.sin(2 * np.pi * hours / 24.0)
df_atacama['HOUR_COS'] = np.cos(2 * np.pi * hours / 24.0)
df_atacama['MONTH_SIN'] = np.sin(2 * np.pi * months / 12.0)
df_atacama['MONTH_COS'] = np.cos(2 * np.pi * months / 12.0)

df_atacama['Latitude'] = -24.09
df_atacama['Longitude'] = -69.92
df_atacama['Tilt'] = 20.0

FINAL_COLUMNS = [
    'utctime', 'Power_W', 'Power_PU',
    'Irradiance_POA', 'GHI', 'Temp_Ambient', 'Wind_Speed', 'Humidity',
    'HOUR_SIN', 'HOUR_COS', 'MONTH_SIN', 'MONTH_COS',
    'Latitude', 'Longitude', 'Tilt'
]
df_atacama = df_atacama[FINAL_COLUMNS]

## Save

In [ ]:
SAVE_PATH = OUT_DIR / 'Atacama.csv'
df_atacama.to_csv(SAVE_PATH, index=False)